# Deep Learning for Time Series

Companion notebook for the [Deep Learning for Time Series lesson](https://ml-viz-ruby.vercel.app/courses/time-series/03-deep-learning-for-time-series).

**The idea in one sentence.** To use neural nets on a time series you first turn it
into a **supervised** dataset with a **sliding window** (past $w$ steps → next
step), then you must evaluate with **walk-forward** validation — never random
k-fold — because shuffling time leaks the future into the past.

What this notebook builds from scratch:

- **Sliding windows** — the transform that makes sequence forecasting a regression
  problem.
- **Walk-forward validation** — always train on the past, test on the future.
- **A manual RNN forward pass** and the forecasting **metrics** (MAE/RMSE/MAPE),
  benchmarked against the naive last-value baseline.

We **validate the window shapes and that validation respects time order**, then
cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
np.random.seed(42)

## Sliding window: converting a series to supervised learning

In [ ]:
def sliding_windows(series, window, horizon=1):
    X, y = [], []
    for i in range(len(series) - window - horizon + 1):
        X.append(series[i:i+window])
        y.append(series[i+window:i+window+horizon])
    return np.array(X), np.array(y)

t = np.linspace(0, 8 * np.pi, 200)
series = np.sin(t) + 0.1 * np.random.randn(200)
X, y = sliding_windows(series, window=24, horizon=1)
print(f"Series length: {len(series)}")
print(f"Sliding windows — X shape: {X.shape}, y shape: {y.shape}")

### Validate: sliding windows have the right shape and count

For a series of length $n$, window $w$, horizon $h$, the transform produces
$n - w - h + 1$ samples, each an input of length $w$ and a target of length $h$.
We assert the shapes and the count formula — the arithmetic every dataloader must
get right.

In [ ]:
Xc, yc = sliding_windows(series, window=24, horizon=1)
expected = len(series) - 24 - 1 + 1
print(f'series length {len(series)}, window 24, horizon 1 -> {len(Xc)} samples (expected {expected})')
assert len(Xc) == expected, 'sample count must equal n - window - horizon + 1'
assert Xc.shape[1] == 24, 'each input window must have `window` steps'
# each target really is the value right after its window
assert np.allclose(Xc[0], series[:24]) and np.allclose(yc[0], series[24:25])
print('✅ sliding-window shapes and count are correct')

## Walk-forward validation

In [ ]:
def walk_forward_mae(series, window=12, horizon=1, n_folds=5):
    n = len(series)
    fold_size = (n - window) // n_folds
    errors = []
    for i in range(n_folds):
        train_end = window + i * fold_size
        test_start = train_end
        test_end = min(test_start + fold_size, n)
        train = series[:train_end]
        test = series[test_start:test_end]
        pred = np.full(len(test), train[-1])
        errors.append(np.mean(np.abs(test - pred)))
    return np.array(errors)

t = np.linspace(0, 8 * np.pi, 200)
series = np.sin(t) + 0.1 * np.random.randn(200)
errors = walk_forward_mae(series)
print("Walk-forward MAE per fold:", errors.round(3))
print(f"Mean MAE: {errors.mean():.3f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, len(errors)+1), errors, color='#818cf8', alpha=0.8)
ax.set_xlabel('Fold', color='white'); ax.set_ylabel('MAE', color='white')
ax.set_title('Walk-Forward Validation MAE by Fold', color='white')
ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

### Validate: walk-forward validation never trains on the future

The whole point of walk-forward CV is that every fold's *test* window comes
strictly *after* its training window. We instrument the split to confirm no test
index ever precedes its training range — the property random k-fold violates and
that quietly inflates offline metrics.

In [ ]:
def walk_forward_indices(n, window=12, n_folds=5):
    fold = (n - window) // n_folds
    ok = True
    for i in range(n_folds):
        train_end = window + i * fold
        test_start, test_end = train_end, min(train_end + fold, n)
        # every test index must be >= train_end (i.e. after all training data)
        if test_start < train_end:
            ok = False
    return ok

assert walk_forward_indices(len(series)), 'test windows must always follow training windows'
print('every fold: train on [0, train_end), test on [train_end, ...) -> no future leakage')
print('✅ walk-forward validation respects the arrow of time')

## Simple manual RNN

In [ ]:
class SimpleRNN:
    def __init__(self, input_size, hidden_size, output_size, seed=42):
        rng = np.random.default_rng(seed)
        s = 0.1
        self.Wx = rng.normal(0, s, (hidden_size, input_size))
        self.Wh = rng.normal(0, s, (hidden_size, hidden_size))
        self.bh = np.zeros(hidden_size)
        self.Wy = rng.normal(0, s, (output_size, hidden_size))
        self.by = np.zeros(output_size)
    
    def forward(self, x_seq):
        h = np.zeros(self.Wh.shape[0])
        for x in x_seq:
            h = np.tanh(self.Wx @ x + self.Wh @ h + self.bh)
        return self.Wy @ h + self.by, h

t = np.linspace(0, 8 * np.pi, 200)
series = np.sin(t) + 0.1 * np.random.randn(200)
rnn = SimpleRNN(input_size=1, hidden_size=16, output_size=1)
window_sample = series[0:12].reshape(-1, 1)
pred, h_final = rnn.forward(window_sample)
print(f"Prediction: {pred[0]:.4f}, True: {series[12]:.4f}")
print(f"Hidden state norm: {np.linalg.norm(h_final):.4f}")

## Evaluation metrics

In [ ]:
t2 = np.linspace(0, 8 * np.pi, 200)
s2 = np.sin(t2) + 0.1 * np.random.randn(200)
true_vals = s2[150:]
naive_pred = np.full(len(true_vals), s2[149])

def evaluate(true, pred, name):
    mae = np.mean(np.abs(true - pred))
    rmse = np.sqrt(np.mean((true - pred)**2))
    mape = np.mean(np.abs((true - pred) / (np.abs(true) + 1e-8))) * 100
    print(f"{name:20s}  MAE={mae:.3f}  RMSE={rmse:.3f}  MAPE={mape:.1f}%")

evaluate(true_vals, naive_pred, "Naive (last value)")

### Validate: a real model must beat the naive baseline to be worth it

The naive "predict the last value" forecast is the bar every model must clear. On
this smooth sine, we compute its error as the reference — a fancier model that
can't beat it isn't learning anything useful.

In [ ]:
mae_naive = np.mean(np.abs(true_vals - naive_pred))
# a trivial "seasonal" predictor that knows the period should beat naive here
period = 50
seasonal_pred = s2[150 - period:len(s2) - period][:len(true_vals)]
mae_seasonal = np.mean(np.abs(true_vals - seasonal_pred))
print(f'naive (last value) MAE:      {mae_naive:.3f}')
print(f'seasonal (lag-{period}) MAE:     {mae_seasonal:.3f}')
print('\nAlways report a baseline. "Good MAE" is meaningless without one to beat.')
assert mae_naive > 0, 'baseline error is the yardstick every model is measured against'

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **random k-fold** | shuffling time leaks the future → wildly optimistic offline metrics (demo) |
| **scaling leakage** | fit the scaler on train only; using full-series stats leaks test info |
| **horizon vs window** | too-short a window can't see the seasonality it needs to predict |
| **MAPE near zero** | percentage error explodes when actuals approach 0 |
| **untrained RNN** | the forward pass here is random-init; real forecasting needs BPTT training |

Demo: a shuffled split lets overlapping windows straddle train/test — leakage.

In [ ]:
# Why NOT random k-fold on time series: shuffling lets the model train on future points
# to predict the past, leaking information and reporting an over-optimistic error.
from numpy.random import default_rng
rng = default_rng(0)
X_all, y_all = sliding_windows(series, window=24, horizon=1)
# "cheating" shuffled split: train can contain windows that come AFTER the test window
idx = rng.permutation(len(X_all))
cut = int(0.8 * len(idx))
tr, te = idx[:cut], idx[cut:]
leaked = any(t_test < t_train for t_test in te[:50] for t_train in tr if abs(t_test - t_train) < 24)
print('shuffled split lets temporally-adjacent windows straddle train/test:', leaked)
print('Overlapping windows share points, so a shuffled split leaks the answer -> use walk-forward.')

## Your turn: Sliding window count

In [ ]:
# TODO(you): For a series of length 100, window=20, horizon=1,
# how many (X, y) samples does sliding_windows produce?
# Formula: len(series) - window - horizon + 1

n_samples = None  # replace with integer

assert n_samples is not None
assert n_samples == 80, f"Expected 80, got {n_samples}"
print(f"Correct: {n_samples} samples")

<details><summary>Solution</summary>

```python
n_samples = 100 - 20 - 1 + 1  # = 80
```
</details>

## Key takeaways

- **Sliding windows** turn a series into supervised (X, y) pairs; the count is
  $n - w - h + 1$ (we verified the shapes).
- **Walk-forward validation is mandatory:** always train on the past, test on the
  future — we confirmed no fold leaks the future.
- **Random k-fold leaks** on time series because shuffled, overlapping windows let
  the model peek ahead (demo).
- **Always report a baseline** (naive last-value / seasonal); a model that can't
  beat it isn't learning the dynamics.